# IEEE 118-Bus System - SC-DCOPF with Optimal Transmission Switching (OTS-SC-DCOPF)

Extension of the SC-DCOPF (`../sc-dcopf/sc-dcopf.ipynb`) with a **binary switching decision**
$z_{ij}$ per branch. The topology is chosen **once** (preventive switching: $z_{ij}$ has no
scenario index) and must keep every operating state $c \in \mathcal{C} = \{0\} \cup
\mathcal{C}_T \cup \mathcal{C}_G$ feasible.

$$\min_{P,f,\theta,z} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

subject to nodal balance (3.9), generator limits (3.10), switching variable (3.11), thermal
limits (3.12), the big-M DC flow law (3.13)/(3.14), the out-of-service flow (3.15), and the
reference-bus constraint (3.16).

All input data is read from `processed/`. Per-unit on a 100 MVA base.

**Why this is harder than the SC-DCOPF:** in the SC-DCOPF the scenarios share no variable, so
the problem decomposes into one DCOPF per state. Here $z_{ij}$ is shared by all states, which
couples them, and the problem becomes a MILP.

**Build strategy:** same as the SC-DCOPF. Start with the base case only ($\mathcal{C}=\{0\}$,
plain OTS-DCOPF), then switch on $\mathcal{C}_T$ and $\mathcal{C}_G$.

## 1. Imports and processed data

In [162]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [163]:
# This notebook lives in a model subfolder (e.g. project/ots-sc-dcopf/), while the
# shared processed data lives one level up. ROOT points to the project data root so paths
# survive folder reorganizations (change here only).
# SOURCE picks which processed dataset to use: 'matpower' or 'scuc'.
ROOT   = ".."
SOURCE = "matpower"   # 'matpower' or 'scuc'
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")

base_mva = 100 MVA | slack bus = 69
generators = 19 | buses = 118 | branches = 186


## 2. Index sets and branch-oriented structures

Same sets and lookups as the SC-DCOPF. The nodal balance (3.9) is written per bus using the
sets of **outgoing** lines $\Omega_i^+$ (branches with `from_bus == i`) and **incoming** lines
$\Omega_i^-$ (`to_bus == i`), plus the generators attached to each bus $\mathcal{G}_i$.

In [164]:
# --- sets ---
set_N = loads["bus"].tolist()          # buses      (N)
set_G = gens["gen_id"].tolist()        # generators (G)
set_L = branches["branch_id"].tolist() # branches   (L)

# --- per-element parameters ---
par_pd     = dict(zip(loads["bus"],          loads["pd_pu"]))      # P_{d,i}
par_g_bus  = dict(zip(gens["gen_id"],        gens["bus"]))         # bus of generator g
par_pmin   = dict(zip(gens["gen_id"],        gens["pmin_pu"]))     # P_g^min
par_pmax   = dict(zip(gens["gen_id"],        gens["pmax_pu"]))     # P_g^max
par_cost   = dict(zip(gens["gen_id"],        gens["c1"]))          # c_g  ($/MWh)
par_from   = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to     = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x      = dict(zip(branches["branch_id"], branches["x_pu"]))    # x_{ij}
par_rate   = dict(zip(branches["branch_id"], branches["rate_pu"])) # F_{ij}^max
par_label  = dict(zip(branches["branch_id"], branches["label"]))   # 'from-to' label

# --- subset G_i : generators connected to each bus ---
set_gens_at = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)

# --- subsets Omega_i^+ (leaving bus i) and Omega_i^- (entering bus i) ---
set_out_lines = {i: [] for i in set_N}   # Omega_i^+  (from_bus == i)
set_in_lines  = {i: [] for i in set_N}   # Omega_i^-  (to_bus == i)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

# --- integrity checks ---
assert SLACK_BUS in set_N, "Slack bus not found among buses."
assert sum(len(v) for v in set_gens_at.values()) == len(set_G)
assert sum(len(v) for v in set_out_lines.values()) == len(set_L)

print(f"|N| = {len(set_N)} buses | |G| = {len(set_G)} generators | |L| = {len(set_L)} branches")
print(f"Bus {SLACK_BUS} (slack): gens={set_gens_at[SLACK_BUS]}, "
      f"lines out={len(set_out_lines[SLACK_BUS])}, lines in={len(set_in_lines[SLACK_BUS])}")

|N| = 118 buses | |G| = 19 generators | |L| = 186 branches
Bus 69 (slack): gens=[12], lines out=3, lines in=3


## 3. Operating states $\mathcal{C}$, availability $N1$, and probabilities $\rho$

Same scenario construction as the SC-DCOPF: $\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup
\mathcal{C}_G$, with $\mathcal{C}_T$ using **non-radial lines only** and $\mathcal{C}_G$ the
real generators. Scenario ids follow the same order (base, lines, generators), so results can
be compared one-to-one with the SC-DCOPF.

In [165]:
INCLUDE_CT = True   # transmission contingencies (non-radial lines)
INCLUDE_CG = True   # generator contingencies

# --- run identity: model / formulation / (config comes from the switches above) ---
MODEL       = "ots-sc-dcopf"
FORMULATION = "redispatch_only"  # redispatch_only | redispatch_with_exclusions | with_load_shedding

# Max number of branches the optimizer may open at once: sum(1 - z) <= K. None = unlimited.
# Part of the run identity, so each K gets its own results folder (K_inf when unlimited).
MAX_SWITCHED = 3

# The set of contingencies to drop is TIED to the formulation, so choosing the formulation
# automatically sets the right behaviour (no risk of mismatched settings).
# The excluded elements were identified by ../sc-dcopf/vulnerability.ipynb (no corrective
# solution on the full topology). With switching they may become feasible: run
# 'redispatch_only' to test it.
if FORMULATION == "redispatch_with_exclusions":
    EXCLUDE = [("line", 7),    # line 8-5
               ("line", 50),   # line 38-37
               ("gen", 0)]     # generator at bus 10 (505 MW)
else:  # 'redispatch_only' (may be infeasible) or 'with_load_shedding' (handled by shedding)
    EXCLUDE = []

# Eligible contingency elements (drop the excluded ones)
CT_lines = [l for l in branches.loc[~branches["is_bridge"], "branch_id"].tolist()
            if ("line", l) not in EXCLUDE]                 # non-radial, not excluded
CG_gens  = [g for g in set_G if ("gen", g) not in EXCLUDE] # real generators, not excluded

# Build the scenario list. Each scenario knows which element (if any) is out of service.
scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None,
              "desc": "base case (all in service)"}]

if INCLUDE_CT:
    for l in CT_lines:
        scenarios.append({"id": len(scenarios), "type": "line",
                          "out_line": l, "out_gen": None, "desc": f"line {l} out"})
if INCLUDE_CG:
    for g in CG_gens:
        scenarios.append({"id": len(scenarios), "type": "gen",
                          "out_line": None, "out_gen": g, "desc": f"gen {g} out"})

set_C = [s["id"] for s in scenarios]     # C : operating states
scen  = {s["id"]: s for s in scenarios}  # lookup by scenario id

# N1 availability (0 = out of service, 1 = in service)
def line_on(l, c):
    return 0 if scen[c]["out_line"] == l else 1

def gen_on(g, c):
    return 0 if scen[c]["out_gen"] == g else 1

print(f"INCLUDE_CT={INCLUDE_CT} | INCLUDE_CG={INCLUDE_CG} | excluded {len(EXCLUDE)} contingencies")
print(f"Eligible: |C_T|={len(CT_lines)} non-radial lines, |C_G|={len(CG_gens)} generators")
print(f"|C| = {len(set_C)} operating states (1 base + {len(set_C) - 1} contingencies)")

# Contingency configuration name (from the switches), and the output folder built as
# results/<source>/<formulation>/<config>/<K>/ so results never overwrite a different run.
config = {
    (False, False): "basecase",       # C = {0}
    (True,  True):  "contingencies",  # base + C_T + C_G
    (True,  False): "cont_trans",     # base + C_T (lines only)
    (False, True):  "cont_gen",       # base + C_G (generators only)
}[(INCLUDE_CT, INCLUDE_CG)]

k_tag   = "K_inf" if MAX_SWITCHED is None else f"K{MAX_SWITCHED}"
OUT_DIR = f"results/{SOURCE}/{FORMULATION}/{config}/{k_tag}"
print(f"MODEL={MODEL} | FORMULATION={FORMULATION} | config={config} | K={MAX_SWITCHED}")
print(f"output -> {OUT_DIR}")

INCLUDE_CT=True | INCLUDE_CG=True | excluded 0 contingencies
Eligible: |C_T|=177 non-radial lines, |C_G|=19 generators
|C| = 197 operating states (1 base + 196 contingencies)
MODEL=ots-sc-dcopf | FORMULATION=redispatch_only | config=contingencies | K=3
output -> results/matpower/redispatch_only/contingencies/K3


### 3.1 Scenario probabilities $\rho_c$

Same per-type failure weights as the SC-DCOPF, so expected costs are directly comparable.

In [166]:
# Per-element failure probabilities: every generator contingency has the same weight
# RHO_GEN, every line contingency the same RHO_LINE. The base case takes the remainder.
RHO_GEN  = 1e-2   # probability of a single generator failure
RHO_LINE = 1e-3   # probability of a single transmission (line) failure

par_rho = {}
for c in set_C:
    s = scen[c]
    if s["type"] == "gen":
        par_rho[c] = RHO_GEN
    elif s["type"] == "line":
        par_rho[c] = RHO_LINE
    # base case is filled below

# the base case takes the remaining probability: rho_0 = 1 - sum of all contingency rhos
rho_cont_sum = sum(par_rho.values())
par_rho[0] = 1.0 - rho_cont_sum

assert par_rho[0] > 0, (
    f"Contingency probabilities sum to {rho_cont_sum:.4f} (>1): lower RHO_GEN/RHO_LINE."
)
assert abs(sum(par_rho.values()) - 1.0) < 1e-9, "rho must sum to 1."
print(f"rho_gen={RHO_GEN}, rho_line={RHO_LINE}")
print(f"rho[0] (base) = {par_rho[0]:.4f}  (contingencies sum to {rho_cont_sum:.4f})")
print(f"sum of rho    = {sum(par_rho.values()):.6f}")

rho_gen=0.01, rho_line=0.001
rho[0] (base) = 0.6330  (contingencies sum to 0.3670)
sum of rho    = 1.000000


### 3.2 Big-M and solver settings

When a branch is switched off ($z_{ij}=0$), (3.13)/(3.14) must stop coupling the angles of its
two ends, so $M$ has to be at least as large as the "virtual flow" $|\theta_i-\theta_j|/x_{ij}$
across the open branch. We use a **per-branch** constant

$$M_{ij} = \frac{\Delta\theta^{\max}}{x_{ij}}$$

instead of a single $M$: reactances range from 0.004 to 0.41 p.u., so a single $M$ sized for
the smallest $x$ would be ~100x larger than needed on most branches, which weakens the LP
relaxation and slows the MILP.

$\Delta\theta^{\max}$ is the largest angle difference allowed across an **open** branch. If it
is too small it cuts off valid topologies; the check in Section 8 warns when it becomes
binding (then increase it and re-run).

The solver settings below do not change the model, only how Gurobi searches it. On the full
N-1 set Gurobi spent its whole time at the root node without finding any feasible topology,
so the defaults push it towards **finding a feasible solution first**:
- `MIPFocus = 1` - prioritize finding feasible solutions over proving optimality.
- `NoRelHeurTime` - seconds of the 'no relaxation' heuristic, which looks for feasible
  solutions before the (slow) root relaxation is solved.
- `Cuts = 1` - moderate cut generation (the root cut loop was slow and barely moved the bound).
- `COMPUTE_IIS` - the IIS of this MILP can take very long; turn it off to just report infeasibility.

In [167]:
DTHETA_MAX = np.pi/2   # [rad] max angle difference across an open branch (big-M sizing)
par_M = {l: DTHETA_MAX / par_x[l] for l in set_L}   # M_ij [p.u.]

# MILP settings: OTS on 118 buses can take long to *prove* optimality even when a good
# solution is found early, so cap the run and report the remaining gap.
MIP_GAP    = 1e-4   # relative optimality gap
TIME_LIMIT = 3600   # [s]
MIP_FOCUS  = 1      # 0 = balanced (Gurobi default), 1 = find feasible solutions first
NO_REL_HEUR_TIME = 300   # [s] 'no relaxation' heuristic before the root LP (0 = off)
CUTS       = 1      # -1 = automatic (Gurobi default), 0 = off, 1 = moderate, 2 = aggressive
# The IIS of this MILP is not informative: z is shared by every scenario, so it ends up
# containing all of them (and took 900 s on K = 1). Keep it off unless debugging a small case.
COMPUTE_IIS = False

print(f"DTHETA_MAX = {np.degrees(DTHETA_MAX):.0f} deg | "
      f"M_ij in [{min(par_M.values()):.1f}, {max(par_M.values()):.1f}] p.u.")

DTHETA_MAX = 90 deg | M_ij in [3.8, 387.9] p.u.


## 4. Model and decision variables

In [168]:
# create empty model
model = gp.Model("OTS-SC-DCOPF_118bus")
model.setParam("OutputFlag", 1)      # set to 1 to show the solver logs so I can debug the code
model.setParam("DualReductions", 1)  # so an infeasible model is reported as INFEASIBLE (not INF_OR_UNBD), enabling IIS
model.setParam("MIPGap", MIP_GAP)
model.setParam("TimeLimit", TIME_LIMIT)
model.setParam("MIPFocus", MIP_FOCUS)
model.setParam("NoRelHeurTime", NO_REL_HEUR_TIME)
model.setParam("Cuts", CUTS)

# create decision variables using addVar with free bounds to be limited by the constraints
var_p = {
    (g, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"p[{g},{c}]")
    for g in set_G for c in set_C # for each gen in set G AND each contingency in set c we create one P variable
}
var_f = {
    (l, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"f[{l},{c}]")
    for l in set_L for c in set_C # for each line in set L AND each contingency in set c we create one flow variable
}
var_theta = {
    (i, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"theta[{i},{c}]")
    for i in set_N for c in set_C # for each bus in set N AND each contingency in set c we create one theta variable
}

# (3.11) switching variable: ONE binary per branch, shared by every scenario (preventive switching)
var_z = {
    l: model.addVar(vtype=GRB.BINARY, name=f"z[{l}]")
    for l in set_L
}

print(f"Created {len(var_p)} p, {len(var_f)} f, {len(var_theta)} theta, {len(var_z)} z variables.")

Set parameter OutputFlag to value 1
Set parameter DualReductions to value 1
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 3600
Set parameter MIPFocus to value 1
Set parameter NoRelHeurTime to value 300
Set parameter Cuts to value 1
Created 3743 p, 36642 f, 23246 theta, 186 z variables.


### Switchable branches

Only **non-radial** branches may be switched, the same set used for $\mathcal{C}_T$. A radial
branch (bridge) is the only path to part of the network, so opening it would isolate buses;
those branches are kept closed:

$$z_{ij} = 1, \qquad \forall (i,j) \in \mathcal{L} \setminus \mathcal{L}^{S}$$

where $\mathcal{L}^{S}$ is the set of non-radial branches. Note this does **not** fully rule out
isolation: opening *several* non-radial branches together can still cut a bus off. The
connectivity check in Section 8.1 reports it if that happens.

**Switching limit (optional).** At most $K$ branches may be open at the same time:

$$\sum_{(i,j) \in \mathcal{L}^{S}} (1 - z_{ij}) \le K$$

$K = 0$ recovers the SC-DCOPF with fixed topology. Set `MAX_SWITCHED = None` to drop it.

In [169]:
set_LS = branches.loc[~branches["is_bridge"], "branch_id"].tolist()   # switchable branches (L^S)

for l in set_L:
    if l not in set_LS:
        model.addConstr(var_z[l] == 1, name=f"zfix[{l}]")

print(f"|L^S| = {len(set_LS)} switchable branches | "
      f"{len(set_L) - len(set_LS)} radial branches fixed closed: "
      f"{[par_label[l] for l in set_L if l not in set_LS]}")

if MAX_SWITCHED is not None:
    model.addConstr(gp.quicksum(1 - var_z[l] for l in set_LS) <= MAX_SWITCHED, name="max_switched")
    print(f"Switching limit: at most K = {MAX_SWITCHED} branches open")
else:
    print("Switching limit: none (K unlimited)")

|L^S| = 177 switchable branches | 9 radial branches fixed closed: ['8-9', '9-10', '71-73', '85-86', '86-87', '110-111', '110-112', '68-116', '12-117']
Switching limit: at most K = 3 branches open


## 5. Objective

$$\min_{P,f,\theta,z} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

Switching has no cost of its own; $z$ affects the objective only through the dispatch it allows.

In [170]:
model.setObjective(
    gp.quicksum(par_rho[c] * par_cost[g] * var_p[(g, c)] * BASE_MVA # par_cost is in $/MWh and var_p is in p.u., so we multiply by BASE_MVA to get $/h
                for g in set_G for c in set_C),
    GRB.MINIMIZE,
)

## 6. Constraints

Each block below is one equation of the formulation, written for every scenario `c`.

### Energy Balance Constraints

$$\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{j \in \Omega_i^+} f_{ij,c} - \sum_{j \in \Omega_i^-} f_{ji,c} \qquad (3.9)$$

In [171]:
for c in set_C:
    for i in set_N:
        gen_i = gp.quicksum(var_p[(g, c)] for g in set_gens_at[i])   # sum over G_i
        out_i = gp.quicksum(var_f[(l, c)] for l in set_out_lines[i]) # Omega_i^+
        in_i  = gp.quicksum(var_f[(l, c)] for l in set_in_lines[i])  # Omega_i^-
        model.addConstr(gen_i - par_pd[i] == out_i - in_i, name=f"bal[{i},{c}]")

### Generator Limits Constraints

$$P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c} \qquad (3.10)$$

When a generator is out (`gen_on == 0`) both bounds collapse to 0, forcing $P_{g,c}=0$.

In [172]:
for c in set_C:
    for g in set_G:
        avail = gen_on(g, c)
        model.addConstr(var_p[(g, c)] >= par_pmin[g] * avail, name=f"pmin[{g},{c}]")
        model.addConstr(var_p[(g, c)] <= par_pmax[g] * avail, name=f"pmax[{g},{c}]")

### Thermal Transmission Limits

$$-F_{ij}^{\max} z_{ij} \le f_{ij,c} \le F_{ij}^{\max} z_{ij} \qquad (3.12)$$

A switched-off branch ($z_{ij}=0$) carries no flow in any scenario.

In [173]:
for c in set_C:
    for l in set_L:
        model.addConstr(var_f[(l, c)] >= -par_rate[l] * var_z[l], name=f"fmin[{l},{c}]")
        model.addConstr(var_f[(l, c)] <=  par_rate[l] * var_z[l], name=f"fmax[{l},{c}]")

### DC Branch Power Flow Equations

$$f_{ij,c} - \tfrac{1}{x_{ij}}(\theta_{i,c} - \theta_{j,c}) + M_{ij}(1 - z_{ij}) \ge 0 \qquad (3.13)$$
$$f_{ij,c} - \tfrac{1}{x_{ij}}(\theta_{i,c} - \theta_{j,c}) - M_{ij}(1 - z_{ij}) \le 0 \qquad (3.14)$$
$$f_{ij,c} = 0 \;\; (N1_{ij,c}=0) \qquad (3.15)$$

(3.13)/(3.14) apply to branches in service in scenario `c`. With $z_{ij}=1$ they collapse to
the DC flow law; with $z_{ij}=0$ they are relaxed by $M_{ij}$ and the angles decouple. A branch
out by contingency gets (3.15) and no angle coupling, as in the SC-DCOPF.

In [174]:
for c in set_C:
    for l in set_L:
        i, j = par_from[l], par_to[l]
        if line_on(l, c) == 1:
            dc_law = var_f[(l, c)] - (1.0 / par_x[l]) * (var_theta[(i, c)] - var_theta[(j, c)])
            model.addConstr(dc_law + par_M[l] * (1 - var_z[l]) >= 0, name=f"dcflow_lo[{l},{c}]")  # (3.13)
            model.addConstr(dc_law - par_M[l] * (1 - var_z[l]) <= 0, name=f"dcflow_hi[{l},{c}]")  # (3.14)
        else:
            model.addConstr(var_f[(l, c)] == 0.0, name=f"dcflow_off[{l},{c}]")                   # (3.15)

### Reference Bus Constraint

$$\theta_{\mathrm{ref},c} = 0 \qquad (3.16)$$

In [175]:
for c in set_C:
    model.addConstr(var_theta[(SLACK_BUS, c)] == 0.0, name=f"ref[{c}]")

print(f"Model built: {model.NumVars} variables ({model.NumBinVars} binary), "
      f"{model.NumConstrs} constraints.")

Model built: 0 variables (0 binary), 0 constraints.


## 7. Solve

In [176]:
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  3600
MIPFocus  1
NoRelHeurTime  300
Cuts  1

Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Optimize a model with 177330 rows, 63817 columns and 523361 nonzeros (Min)
Model fingerprint: 0x1715e4bd
Model has 3743 linear objective coefficients
Variable types: 63631 continuous, 186 integer (186 binary)
Coefficient statistics:
  Matrix range     [7e-01, 4e+02]
  Objective range  [1e+00, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-02, 4e+02]

Presolve removed 92852 rows and 26205 columns
Presolve time: 3.18s
Presolved: 84478 rows, 37612 columns, 278870 nonzeros
Variable types: 37526 continuous, 86 integer (86 binary)
Starting NoRel heuristic
Found phase-1

## 8. Results

With $\mathcal{C}=\{0\}$ this is a plain OTS-DCOPF: the cost must be **lower than or equal to**
the SC-DCOPF base case ($93,152.38/h), since keeping every branch closed ($z=1$) is still a
feasible choice. The gap between the two is the value of switching.

In [177]:
# A solution exists if the solver proved optimality, or stopped (time limit / manual interrupt)
# with an incumbent
has_solution = model.SolCount > 0 and model.status in (GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.INTERRUPTED)
status_name  = {GRB.OPTIMAL: "OPTIMAL", GRB.TIME_LIMIT: "TIME LIMIT",
                GRB.INTERRUPTED: "INTERRUPTED", GRB.INFEASIBLE: "INFEASIBLE"}.get(model.status, str(model.status))

if has_solution:
    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    switched   = [l for l in set_L if var_z[l].X < 0.5]   # branches opened by the optimizer
    print("=" * 55)
    print("  OTS-SC-DCOPF SOLUTION - IEEE 118-Bus")
    print("=" * 55)
    print(f"  Status          : {status_name}"
          f" (gap {100 * model.MIPGap:.4f}%)")
    print(f"  Expected cost   : ${model.ObjVal:,.2f} /h")
    print(f"  Best bound      : ${model.ObjBound:,.2f} /h")
    print(f"  Total load      : {total_load:.2f} MW")
    print(f"  Total generation: {total_gen:.2f} MW (base case)")
    print(f"  Scenarios       : {len(set_C)}")
    print(f"  Switching limit : K = {MAX_SWITCHED}")
    print(f"  Lines switched  : {len(switched)} -> {[par_label[l] for l in switched]}")
elif model.status == GRB.INFEASIBLE and not COMPUTE_IIS:
    print(f"INFEASIBLE (K = {MAX_SWITCHED}). IIS skipped (COMPUTE_IIS = False).")
elif model.status == GRB.INFEASIBLE:
    import os, re
    print("INFEASIBLE. Computing IIS to locate the conflict...")
    model.computeIIS()

    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)

    # 1) full IIS file (Gurobi native) - the ground truth
    model.write(f"{OUT}/iis_full.ilp")

    # 2) deterministic per-scenario summary, parsed from constraint names like 'fmax[12,49]'
    #    (name pattern: <type>[..,<scenario_id>]). Only constraints flagged by the IIS are kept.
    pat = re.compile(r"^([a-zA-Z_]+)\[.*,(\d+)\]$")
    recs = []
    for con in model.getConstrs():
        if con.IISConstr:
            mm = pat.match(con.ConstrName)
            if mm:
                recs.append({"scenario_id": int(mm.group(2)), "ctype": mm.group(1)})
    iis_df = pd.DataFrame(recs)
    summary = (iis_df.groupby("scenario_id")
               .agg(n_constraints=("ctype", "size"),
                    constraint_types=("ctype", lambda s: ",".join(sorted(set(s)))))
               .reset_index())
    summary["scenario_desc"] = summary["scenario_id"].map(lambda c: scen[c]["desc"])

    # translate each conflicting scenario to the physical element that is out of service,
    # so 'scenario 49' is immediately readable as 'line 38-37' (or 'gen bus 10').
    def out_element(c):
        s = scen[c]
        if s["out_line"] is not None:
            return f"line {par_label[s['out_line']]} (branch_id {s['out_line']})"
        if s["out_gen"] is not None:
            return f"gen bus {par_g_bus[s['out_gen']]} (gen_id {s['out_gen']})"
        return "base case"
    summary["out_element"] = summary["scenario_id"].map(out_element)

    summary = summary[["scenario_id", "scenario_desc", "out_element",
                       "n_constraints", "constraint_types"]]
    summary.to_csv(f"{OUT}/iis_summary.csv", index=False)

    print(f"Saved iis_full.ilp and iis_summary.csv to {OUT}/")
    print(f"Conflicting scenarios: {summary['scenario_id'].tolist()}")
    display(summary)
else:
    print(f"Solver status: {status_name} (no solution available)")

Solver status: TIME LIMIT (no solution available)


### 8.1 Big-M check

For every open branch and every scenario, compare the angle difference across it with
$\Delta\theta^{\max}$. If the largest one reaches $\Delta\theta^{\max}$, the big-M was binding
(it limited the topology), so increase `DTHETA_MAX` and re-run.

Switching can also **isolate** buses (e.g. a bus with only an expensive, idle generator). Their
angle is free (no path to the slack), so it drifts to the big-M bound without meaning
anything; branches touching an isolated bus are therefore left out of the check.

In [178]:
if has_solution:
    # buses still connected to the slack through closed branches (base topology)
    adj = {i: [] for i in set_N}
    for l in set_L:
        if var_z[l].X > 0.5:
            adj[par_from[l]].append(par_to[l])
            adj[par_to[l]].append(par_from[l])
    connected, stack = {SLACK_BUS}, [SLACK_BUS]
    while stack:
        for v in adj[stack.pop()]:
            if v not in connected:
                connected.add(v)
                stack.append(v)
    isolated = [i for i in set_N if i not in connected]
    print(f"Isolated buses: {isolated} "
          f"(load {sum(par_pd[i] for i in isolated) * BASE_MVA:.1f} MW, "
          f"gens {[g for i in isolated for g in set_gens_at[i]]})")

    gaps = [abs(var_theta[(par_from[l], c)].X - var_theta[(par_to[l], c)].X)
            for l in set_L if var_z[l].X < 0.5
            and par_from[l] in connected and par_to[l] in connected
            for c in set_C if line_on(l, c) == 1]
    if gaps:
        worst = max(gaps)
        print(f"max |theta_i - theta_j| across open branches: {np.degrees(worst):.2f} deg "
              f"(limit {np.degrees(DTHETA_MAX):.0f} deg)")
        if worst >= 0.99 * DTHETA_MAX:
            print("WARNING: big-M is binding - increase DTHETA_MAX and re-run.")
        else:
            print("OK: big-M is not binding.")
    else:
        print("No branch was switched off: big-M check not applicable.")

In [179]:
# Base-case dispatch of the active generators
if has_solution:
    rows = []
    for g in set_G:
        mw = var_p[(g, 0)].X * BASE_MVA
        if mw > 0.01:
            rows.append({"gen_id": g, "bus": par_g_bus[g],
                         "p_mw": round(mw, 2),
                         "pmax_mw": round(par_pmax[g] * BASE_MVA, 2),
                         "c1": round(par_cost[g], 3)})
    df_dispatch = pd.DataFrame(rows).sort_values("c1").reset_index(drop=True)
    print(f"Active generators: {len(df_dispatch)}")
    display(df_dispatch)

## 9. Export results for documentation

Saves the solution to `results/` (only when a solution exists). Same files as the SC-DCOPF,
plus the switching decision:
- `summary.csv`      - cost, bound, gap, load, generation, scenario/generator/switching counts
- `dispatch.csv`     - per-generator output and loading (base case)
- `line_loading.csv` - per-line flow, loading and status `z` (base case), sorted by loading
- `angles.csv`       - bus voltage angles (base case)
- `switching.csv`    - the branches opened by the optimizer

In [180]:
import os
import math

if has_solution:
    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)

    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    n_active   = sum(1 for g in set_G if var_p[(g, 0)].X * BASE_MVA > 0.01)
    switched   = [l for l in set_L if var_z[l].X < 0.5]

    # 1) summary
    pd.DataFrame({
        "metric": ["expected_cost_usd_per_h", "best_bound_usd_per_h", "mip_gap",
                   "solver_status", "runtime_s", "total_load_mw", "total_generation_mw",
                   "n_scenarios", "n_active_generators", "max_switched", "n_switched_lines",
                   "dtheta_max_deg"],
        "value":  [round(model.ObjVal, 2), round(model.ObjBound, 2), model.MIPGap,
                   model.status, round(model.Runtime, 1), round(total_load, 2),
                   round(total_gen, 2), len(set_C), n_active,
                   "inf" if MAX_SWITCHED is None else MAX_SWITCHED, len(switched),
                   round(math.degrees(DTHETA_MAX), 1)],
    }).to_csv(f"{OUT}/summary.csv", index=False)

    # 2) per-generator dispatch (base case, c = 0)
    disp = []
    for g in set_G:
        mw   = var_p[(g, 0)].X * BASE_MVA
        pmax = par_pmax[g] * BASE_MVA
        disp.append({"gen_id": g, "bus": par_g_bus[g], "p_mw": round(mw, 2),
                     "pmax_mw": round(pmax, 2),
                     "loading_pct": round(100 * mw / pmax, 1) if pmax > 0 else 0,
                     "c1": round(par_cost[g], 3)})
    pd.DataFrame(disp).sort_values("c1").to_csv(f"{OUT}/dispatch.csv", index=False)

    # 3) per-line loading (base case), with switching status - sorted to reveal congestion
    fl = []
    for l in set_L:
        f    = var_f[(l, 0)].X * BASE_MVA
        fmax = par_rate[l] * BASE_MVA
        fl.append({"branch_id": l, "label": par_label[l], "z": int(round(var_z[l].X)),
                   "f_mw": round(f, 2), "fmax_mw": round(fmax, 2),
                   "loading_pct": round(100 * abs(f) / fmax, 1) if fmax > 0 else 0})
    (pd.DataFrame(fl).sort_values("loading_pct", ascending=False)
        .to_csv(f"{OUT}/line_loading.csv", index=False))

    # 4) bus voltage angles (base case). Slack is fixed at 0; others are relative to it.
    ang = []
    for i in set_N:
        rad = var_theta[(i, 0)].X
        ang.append({"bus": i, "theta_rad": round(rad, 6),
                    "theta_deg": round(math.degrees(rad), 4),
                    "is_slack": i == SLACK_BUS})
    (pd.DataFrame(ang).sort_values("theta_deg")
        .to_csv(f"{OUT}/angles.csv", index=False))

    # 5) switching decision: the opened branches
    pd.DataFrame([{"branch_id": l, "label": par_label[l], "x_pu": par_x[l],
                   "fmax_mw": round(par_rate[l] * BASE_MVA, 2)}
                  for l in switched],
                 columns=["branch_id", "label", "x_pu", "fmax_mw"]
                 ).to_csv(f"{OUT}/switching.csv", index=False)

    print(f"Saved 5 files to {OUT}/ (cost=${model.ObjVal:,.2f}/h, {n_active} active gens, "
          f"{len(switched)} lines switched).")
else:
    print("No solution - nothing exported.")

No solution - nothing exported.


In [181]:
rows = [(par_label[l], c, scen[c]["desc"],
         np.degrees(abs(var_theta[(par_from[l], c)].X - var_theta[(par_to[l], c)].X)))
        for l in set_L if var_z[l].X < 0.5 for c in set_C if line_on(l, c) == 1]
pd.DataFrame(rows, columns=["open_line", "scenario", "desc", "dtheta_deg"]
             ).sort_values("dtheta_deg", ascending=False).head(10)

AttributeError: Unable to retrieve attribute 'X'